```text
* 활용문서 : 종합부동산세법
* ocr 을 돌려서 md 파일로 생성: ./documents/output_docs/real_estate.md
```

```text
                         ┌────────────────────┐
                    ┌───▶│ 과세표준           │───┐
                    │    │                │   │
                    │    └────────────────────┘   │
                    │                             │
┌─────────┐         │    ┌────────────────────┐   │    ┌────────────────┐
│  START  │─────────┼───▶│ 공제금액           │───┼───▶│ 과세표준 계산  │
└─────────┘         │    └────────────────────┘   │    │                │
                    │                             │    └───────┬────────┘
                    │    ┌────────────────────┐   │            │
                    └───▶│ 공정시장가액비율   │───┘            ▼
                         └────────────────────┘        ┌────────────┐
                                                       │ 세율계산    │
                                                       └────────────┘

```

In [1]:
"""
- `주택 수` → 사용자 질문에 있을 것
- `과세표준`
    - 종합부동산세법→ `“주택의 공시가격을 합산한 금액에서 다음 각 호의 금액을 공제한 금액에 부동산 시장의 동향과 재정 여건 등을 정하는 **공정시장가액비율**을 곱한 금액으로 한다.”`
    - `주택의 공시가격을 합산한 금액` → 사용자 질문에 있음
    - `공제한 금액`  → 종합부동산 세법에 있음
        - 세법 : `“대통령령으로 정하는 공정시장가액비율을 곱한 금액으로 한다. 다만, 그 금액이 영보다 작은 경우에는 ~”` → 이걸 벡터 스토어 대상으로 retrieve 해줘야 함
    - `공정시장가액비율` → 대통령령이므로 웹 서치(법안 없음)
"""

'\n- `주택 수` → 사용자 질문에 있을 것\n- `과세표준`\n    - 종합부동산세법→ `“주택의 공시가격을 합산한 금액에서 다음 각 호의 금액을 공제한 금액에 부동산 시장의 동향과 재정 여건 등을 정하는 **공정시장가액비율**을 곱한 금액으로 한다.”`\n    - `주택의 공시가격을 합산한 금액` → 사용자 질문에 있음\n    - `공제한 금액`  → 종합부동산 세법에 있음\n        - 세법 : `“대통령령으로 정하는 공정시장가액비율을 곱한 금액으로 한다. 다만, 그 금액이 영보다 작은 경우에는 ~”` → 이걸 벡터 스토어 대상으로 retrieve 해줘야 함\n    - `공정시장가액비율` → 대통령령이므로 웹 서치(법안 없음)\n'

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
""" [개발 노드 목록]
        과세표준 계산
        공제액
        공정시장가액비율
        과세표준 계산
        세율 계산
"""

'[개발 노드 목록]\n        과세표준 계산\n        공제액\n        공정시장가액비율\n        과세표준 계산\n        세율 계산\n'

In [4]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    query: str
    answer: str  # 세율 계산
    tax_base_equation: str # 과세표준 계산 수식
    tax_deduction: str # 공제액
    market_ratio: str # 공정시장가액비율
    market_ratio: str  # 과세표준 계산

graph_builder = StateGraph(AgentState)

#### 문서 임베딩 및 DB 적재

In [ ]:
# loader활용 시 테이블 구조가 사라진다
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# langchain 의 text splitter : 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 100,
    separators=['\n\n', '\n']
)

loader = UnstructuredMarkdownLoader(
    "../documents/output_docs/real_estate_tax.md", 
    mode="elements", 
    strategy="fast")

# data = loader.load()
document_list = loader.load_and_split(text_splitter=text_splitter) 

C:\Users\Yurim Park\AppData\Local\Temp\ipykernel_25564\3136749120.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredMarkdownLoader


In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

In [ ]:
# 리트리버 생성
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=document_list,
    embedding=embeddings,
    collection_name='real_estate_tax_collection',
    persist_directory='../real_estate_tax_collection'   # 로컬에 설정한 경로로 데이터가 남음 (DB의 역할을 한다)
)

retriever = vector_store.as_retriever(search_kwargs={'k':3})  # 3개 문서 반환

In [ ]:
##########################################
## 테스트 코드
##########################################
from collections import Counter

data = vector_store.get(include=["documents"])
documents = data["documents"]
content_counts = Counter(documents)
duplicate_groups = {
    content: count
    for content, count in content_counts.items()
    if count > 1
}

print(f"전체 레코드 수: {len(documents)}")
print(f"고유 문서 수: {len(content_counts)}")
print(f"중복된 문서 그룹 수: {len(duplicate_groups)}")
print(
    "중복으로 추가된 레코드 수:",
    sum(count - 1 for count in duplicate_groups.values()),
)

전체 레코드 수: 106
고유 문서 수: 99
중복된 문서 그룹 수: 3
중복으로 추가된 레코드 수: 7


#### LLM 선언

In [ ]:
from langsmith import Client

client = Client()

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model='exaone3.5:7.8b', 
    num_predict=512,
)

small_llm = ChatOllama(
    model = 'bge-m3:latest'  # 경량화된 모델
)

# 프롬프트
rag_prompt = client.pull_prompt(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True,
)

#### 노드 생성

In [ ]:
from langchain_core.output_parsers import StrOutputParser

tax_base_equation_chain = rag_prompt | llm | StrOutputParser()

def get_tax_base_equation(state: AgentState) -> str:
    tax_base_equation_chain.invoke()
    return state['tax_base_equation']
    